In [1]:
from pathlib import Path
import pandas as pd
import os

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
training_team_path = Path(os.path.dirname(os.getcwd())) / "data" / "processed" / "training_team_data.parquet"
team_df = pd.read_parquet(training_team_path)

In [3]:
correct_results = team_df[
    ((team_df["elo_win_likelihood"] >= 0.5) & (team_df["result"] == 1))
    | ((team_df["elo_win_likelihood"] < 0.5) & (team_df["result"] == 0))
]
elo_accuracy = len(correct_results) / len(team_df)

In [4]:
# Glicko2 Validation
team_df["gl2_prediction"]  = team_df["gl2_win_likelihood"] >= 0.5
team_df["gl2_correct"] = team_df["gl2_prediction"] == team_df["result"]
gl2_accuracy = team_df["gl2_correct"].mean()

In [5]:
# Plackett-Luce Validation
team_df["pl_prediction"]  = team_df["pl_win_likelihood"] >= 0.5
team_df["pl_correct"] = team_df["pl_prediction"] == team_df["result"]
pl_accuracy = team_df["pl_correct"].mean()

In [6]:
# Trueskill Validation
team_df["trueskill_prediction"]  = team_df["trueskill_win_likelihood"] >= 0.5
team_df["trueskill_correct"] = team_df["trueskill_prediction"] == team_df["result"]
trueskill_accuracy = team_df["trueskill_correct"].mean()

In [7]:
# Whr Validation
whr_accuracy = 0.0
# team_df["whr_prediction"]  = team_df["whr_win_likelihood"] >= 0.5
# team_df["whr_correct"] = team_df["whr_prediction"] == team_df["result"]
# whr_accuracy = team_df["whr_correct"].mean()

In [8]:
diff_leagues_df = team_df[team_df["league_elo_win_likelihood"] != 0.5]
league_correct_results = diff_leagues_df[
    ((diff_leagues_df["league_elo_win_likelihood"] >= 0.5) & (diff_leagues_df["result"] == 1))
    | ((diff_leagues_df["league_elo_win_likelihood"] < 0.5) & (diff_leagues_df["result"] == 0))
]
league_elo_accuracy = len(league_correct_results) / len(diff_leagues_df)

In [9]:
# Side WR Validation
team_df["ema_side_prediction"]  = team_df["side_win_likelihood"] >= 0.5
team_df["ema_side_correct"] = team_df["ema_side_prediction"] == team_df["result"]
ema_side_accuracy = team_df["ema_side_correct"].mean()

In [10]:
# Patch WR Validation
team_df["ema_patch_prediction"]  = team_df["patch_win_likelihood"] >= 0.5
team_df["ema_patch_correct"] = team_df["ema_patch_prediction"] == team_df["result"]
ema_patch_accuracy = team_df["ema_patch_correct"].mean()

In [11]:
# Season WR Validation
team_df["ema_season_prediction"]  = team_df["season_win_likelihood"] >= 0.5
team_df["ema_season_correct"] = team_df["ema_season_prediction"] == team_df["result"]
ema_season_accuracy = team_df["ema_season_correct"].mean()

In [12]:
# Creta e table where for every model we have the accuracy
model_accuracy = pd.DataFrame({
    "Model": ["Elo", "GL2", "PL", "Trueskill", "League Elo", "WHR", "EMA Side", "EMA Patch"],
    "Accuracy": [elo_accuracy, gl2_accuracy, pl_accuracy, trueskill_accuracy, league_elo_accuracy, whr_accuracy, ema_side_accuracy, ema_patch_accuracy]
})

model_accuracy.sort_values("Accuracy", ascending=False, inplace=True)
model_accuracy

,Model,Accuracy
7,EMA Patch,0.657002
4,League Elo,0.643110
2,PL,0.627462
0,Elo,0.626486
3,Trueskill,0.625306
1,GL2,0.624989
6,EMA Side,0.621427
5,WHR,0.000000
